# Acceptance: contact-survey matrices

**This is a sign-off page, not a tutorial.** It exercises roadmap steps 17 and 18
(work package `WP9`). Step 17: `summer4.epi.ContactMatrix`, which loads,
validates and inspects an empirical contact-survey matrix, and `SettingStack`,
the per-setting stack these datasets ship as. Step 18: everything a survey
matrix needs before a model can use it — rebinning onto the model's age
property, reciprocity correction, adaptation to another population, scaling
for interventions, and conversion to the `MixingMatrix` a force of infection
takes. Nothing here asserts; the automated
checks live in `tests/test_contact_matrices.py`.

Run every cell top to bottom (under a minute). Sign off only if every
**What to check** box holds.

| | You are checking that… |
|---|---|
| A | a published survey table loads in participant-by-contact orientation, and the row and column sums mean different things |
| B | a long frame and a per-setting frame load to the same matrix |
| C | bad input is refused with the offending value named |
| D | reciprocity error and mean contacts are measurable against a population |
| E | rebinning onto a model's age property keeps the contacts the survey measured |
| F | symmetrising makes the matrix reciprocal without changing total contacts |
| G | adapting to another population follows the convention you choose |
| H | closing schools from a date changes the epidemic, through the model's own force of infection |
| I | the scale of a setting is a parameter you can calibrate: it runs under `jit` and has a gradient |
| J | the compiled model does not grow with the number of age bands |

**Orientation, fixed once:** `values[i, j]` is the average number of contacts a
member of band `i` (the *participant*) has with members of band `j` (the
*contact*), per person per day.

**Where the data come from.** summer4 ships no survey data (roadmap step 17,
decided 2026-09-29): it ships loaders, and each page embeds the small public
extract it needs, so nothing is fetched when the docs build. The matrix below is
the POLYMOD Great Britain all-settings matrix from Mossong *et al.* (2008),
"Social contacts and mixing patterns relevant to the spread of infectious
diseases", *PLoS Medicine* 5(3): e74, supporting table
[doi:10.1371/journal.pmed.0050074.st005](https://doi.org/10.1371/journal.pmed.0050074.st005),
published under the Creative Commons Attribution licence. It is carried here as
transcribed in the [summer textbook](https://github.com/monash-emu/summer-textbook)
(BSD-2-Clause, Copyright (c) 2022, monash-emu) chapter 17, together with that
textbook's United Kingdom 2006 population by age.

In [ ]:
import math

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4.epi import ContactMatrix, SettingStack

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

# POLYMOD Great Britain, as published: rows are the AGE OF CONTACT, columns the
# AGE GROUP OF PARTICIPANT (the transpose of summer4's orientation).
PUBLISHED = np.array(
    [
        [1.92, 0.65, 0.41, 0.24, 0.46, 0.73, 0.67, 0.83, 0.24, 0.22, 0.36, 0.20, 0.20, 0.26, 0.13],
        [0.95, 6.64, 1.09, 0.73, 0.61, 0.75, 0.95, 1.39, 0.90, 0.16, 0.30, 0.22, 0.50, 0.48, 0.20],
        [0.48, 1.31, 6.85, 1.52, 0.27, 0.31, 0.48, 0.76, 1.00, 0.69, 0.32, 0.44, 0.27, 0.41, 0.33],
        [0.33, 0.34, 1.03, 6.71, 1.58, 0.73, 0.42, 0.56, 0.85, 1.16, 0.70, 0.30, 0.20, 0.48, 0.63],
        [0.45, 0.30, 0.22, 0.93, 2.59, 1.49, 0.75, 0.63, 0.77, 0.87, 0.88, 0.61, 0.53, 0.37, 0.33],
        [0.79, 0.66, 0.44, 0.74, 1.29, 1.83, 0.97, 0.71, 0.74, 0.85, 0.88, 0.87, 0.67, 0.74, 0.33],
        [0.97, 1.07, 0.62, 0.50, 0.88, 1.19, 1.67, 0.89, 1.02, 0.91, 0.92, 0.61, 0.76, 0.63, 0.27],
        [1.02, 0.98, 1.26, 1.09, 0.76, 0.95, 1.53, 1.50, 1.32, 1.09, 0.83, 0.69, 1.02, 0.96, 0.20],
        [0.55, 1.00, 1.14, 0.94, 0.73, 0.88, 0.82, 1.23, 1.35, 1.27, 0.89, 0.67, 0.94, 0.81, 0.80],
        [0.29, 0.54, 0.57, 0.77, 0.97, 0.93, 0.57, 0.80, 1.32, 1.87, 0.61, 0.80, 0.61, 0.59, 0.57],
        [0.33, 0.38, 0.40, 0.41, 0.44, 0.85, 0.60, 0.61, 0.71, 0.95, 0.74, 1.06, 0.59, 0.56, 0.57],
        [0.31, 0.21, 0.25, 0.33, 0.39, 0.53, 0.68, 0.53, 0.55, 0.51, 0.82, 1.17, 0.85, 0.85, 0.33],
        [0.26, 0.25, 0.19, 0.24, 0.19, 0.34, 0.40, 0.39, 0.47, 0.55, 0.41, 0.78, 0.65, 0.85, 0.57],
        [0.09, 0.11, 0.12, 0.20, 0.19, 0.22, 0.13, 0.30, 0.23, 0.13, 0.21, 0.28, 0.36, 0.70, 0.60],
        [0.14, 0.15, 0.21, 0.10, 0.24, 0.17, 0.15, 0.41, 0.50, 0.71, 0.53, 0.76, 0.47, 0.74, 1.47],
    ]
)
BANDS = (*range(0, 75, 5), math.inf)  # 0-4, 5-9, ..., 65-69, 70+
UK_2006 = np.array(
    [3458060, 3556024, 3824317, 3960916, 3911291, 3762213, 4174675, 4695853,
     4653082, 3986098, 3620216, 3892985, 3124676, 2706365, 6961183],
    dtype=float,
)

polymod = ContactMatrix.from_array(PUBLISHED.T, BANDS, source_population=UK_2006)
polymod

## A. The survey loads the right way round

The published table is contact-by-participant, so it is transposed once on the
way in (`PUBLISHED.T`). The heatmap is `polymod.to_pandas()`: participant band
down the side, contact band along the top. The line chart compares the two
sums that are easy to confuse: `total()` (row sums — contacts each participant
reported) and the column sums (how often each band *was reported as a
contact*). Chapter 17 of the textbook points out that the second depends on
how many people are in the contacted band.

In [ ]:
heat = polymod.to_pandas().plot(
    kind="imshow",
    title="A. POLYMOD Great Britain: contacts per participant per day",
    labels={"x": "contact band", "y": "participant band", "color": "contacts"},
)
heat.show()

sums = pd.DataFrame(
    {
        "total(): contacts reported per participant (row sum)": polymod.total(),
        "column sum: times reported as a contact": polymod.values.sum(axis=0),
    },
    index=pd.Index(polymod.labels, name="band"),
)
fig = sums.plot(title="A. Row sums and column sums are different quantities", markers=True)
fig.update_layout(xaxis_title="age band", yaxis_title="contacts per day", legend_title="")
fig.show()
sums.round(2).T

> **What to check**
>
> - The heatmap has **participant** bands down the side. Its brightest cells are on the diagonal at `[5,10)`, `[10,15)` and `[15,20)` (about 6.6–6.9 contacts a day with their own age), and a fainter stripe runs parallel to the diagonal about 30 years off it (parents and children).
> - In the line chart the row sums peak at `[15,20)` (about **15.5**) and are lowest for `[70,inf)` (about **7.3**).
> - The column sums instead **jump up** at the last band, from about **3.9** at `[65,70)` to about **6.8** at `[70,inf)`: the open-ended 70+ band has many more people in it to be contacted. The row sums show no such jump.

## B. Frames and per-setting stacks load to the same matrix

Survey exports are usually long frames, one row per (participant band, contact
band). Below the same numbers are written as one, with `"0-4"`-style labels,
and read back with `ContactMatrix.from_frame`; the rows are shuffled first, so
the loader has to order the bands itself.

Surveys also split contacts by setting. To have a per-setting frame to load,
the next cell splits the matrix into an **illustrative** `"school"` layer —
*an assumption for this page, not survey data*: 60% of contacts among the three
school-age bands `[5,20)` — and `"other"`, the rest. `SettingStack.from_frame`
reads the frame with its `setting` column; `total()` sums the settings back.
The bars show each setting's contacts per participant, stacked; the line is
the all-settings `total()` from part A.

In [ ]:
labels = [f"{lo}-{lo + 4}" for lo in range(0, 70, 5)] + ["70+"]
k = len(labels)
long = pd.DataFrame(
    {
        "participant_age": [labels[i] for i in range(k) for _ in range(k)],
        "contact_age": [labels[j] for _ in range(k) for j in range(k)],
        "contacts": polymod.values.reshape(-1),
    }
).sample(frac=1.0, random_state=1)
from_frame = ContactMatrix.from_frame(
    long, row="participant_age", col="contact_age", value="contacts", source_population=UK_2006
)

school_values = np.zeros_like(polymod.values)
school_values[1:4, 1:4] = 0.6 * polymod.values[1:4, 1:4]
per_setting = pd.concat(
    [
        long.assign(setting="school", contacts=school_values.reshape(-1)[long.index]),
        long.assign(setting="other", contacts=(polymod.values - school_values).reshape(-1)[long.index]),
    ]
)
stack = SettingStack.from_frame(
    per_setting, row="participant_age", col="contact_age", source_population=UK_2006
)

by_setting = pd.DataFrame(
    {name: stack[name].total() for name in stack}, index=pd.Index(polymod.labels, name="band")
)
fig = by_setting.plot(kind="bar", title="B. Contacts per participant, stacked by setting")
fig.add_scatter(x=list(polymod.labels), y=polymod.total(), mode="lines+markers", name="all settings (part A)")
fig.update_layout(xaxis_title="participant band", yaxis_title="contacts per day", barmode="stack")
fig.show()

pd.Series(
    {
        "from_frame == from_array": from_frame == polymod,
        "max |from_frame - from_array|": float(np.max(np.abs(from_frame.values - polymod.values))),
        "settings": list(stack),
        "max |stack.total() - from_array|": float(np.max(np.abs(stack.total().values - polymod.values))),
    },
    name="value",
).to_frame()

> **What to check**
>
> - `from_frame == from_array` is **True** and both differences in the table are exactly **0.0**: the shuffled long frame, with `"0-4"` / `"70+"` labels, loads to the same matrix.
> - The settings are `['school', 'other']`, and in the chart the stacked bars reach the all-settings line exactly in every band. The school layer appears only in the `[5,10)`, `[10,15)` and `[15,20)` bars.

## C. Bad input is refused, naming the offending value

Each row below is a broken input. The table shows the `ValueError` message
each one raises.

In [ ]:
bad_values = polymod.values.copy()
bad_values[3, 7] = -0.2
attempts = {
    "not square": lambda: ContactMatrix.from_array(polymod.values[:, :-1], BANDS),
    "one band edge too many": lambda: ContactMatrix.from_array(polymod.values, (*BANDS[:-1], 75, math.inf)),
    "bands out of order": lambda: ContactMatrix.from_array(polymod.values, (0, 5, 15, 10, *BANDS[4:])),
    "a negative contact rate": lambda: ContactMatrix.from_array(bad_values, BANDS),
    "population of the wrong length": lambda: ContactMatrix.from_array(
        polymod.values, BANDS, source_population=UK_2006[:-1]
    ),
    "an unreadable band label": lambda: ContactMatrix.from_frame(
        long.replace({"participant_age": {"15-19": "teens"}, "contact_age": {"15-19": "teens"}}),
        row="participant_age",
        col="contact_age",
    ),
    "a missing (participant, contact) pair": lambda: ContactMatrix.from_frame(
        long.iloc[1:], row="participant_age", col="contact_age"
    ),
}


def message(attempt):
    try:
        attempt()
    except ValueError as error:
        return str(error)
    return "accepted (this would be a failure)"


pd.set_option("display.max_colwidth", None)
pd.DataFrame({"message": {name: message(attempt) for name, attempt in attempts.items()}})

> **What to check** — every row is refused, and its message names the actual value:
>
> - *not square* names the shape `(15, 14)`;
> - *one band edge too many* says **17** band edges for a 15 × 15 matrix, expected **16**;
> - *bands out of order* names the pair `bands[2] = 15 then bands[3] = 10`;
> - *a negative contact rate* names `values[3, 7] = -0.2`;
> - *population of the wrong length* names shape `(14,)` against the expected `(15,)`;
> - *an unreadable band label* names `'teens'`;
> - *a missing pair* names the pair that is missing.

## D. Reciprocity and mean contacts are measurable

A contact is between two people, so in a closed population the total contacts
band `i` reports with band `j` should equal the total `j` reports with `i`:
$c_{ij} N_i = c_{ji} N_j$. A survey only ever estimates that. The heatmap is
`polymod.reciprocity_error().matrix`, $c_{ij} N_i - c_{ji} N_j$ in millions of
contacts a day, against the UK 2006 population the survey carries as
`source_population`. `reciprocity_error().relative` is one number for the
whole matrix, and `mean_contacts()` is the population-weighted average
contacts per person — the headline number a survey paper reports.

In [ ]:
reciprocity = polymod.reciprocity_error()
error = pd.DataFrame(
    reciprocity.matrix / 1e6,
    index=pd.Index(polymod.labels, name="participant"),
    columns=pd.Index(polymod.labels, name="contact"),
)
fig = error.plot(
    kind="imshow",
    title="D. Reciprocity error c_ij N_i - c_ji N_j (millions of contacts a day)",
    labels={"x": "contact band", "y": "participant band", "color": "millions"},
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
)
fig.show()

pd.Series(
    {
        "relative reciprocity error": round(reciprocity.relative, 3),
        "largest |error| (millions/day)": round(float(np.max(np.abs(error.values))), 2),
        "at (participant, contact)": error.abs().stack().idxmax(),
        "mean contacts per person per day": round(polymod.mean_contacts(), 2),
    },
    name="value",
).to_frame()

> **What to check**
>
> - The heatmap is **antisymmetric**: each cell is the negative of its mirror across the diagonal, and the diagonal is zero (white).
> - The last row (`[70,inf)` participants) is mostly red and the last column mostly blue: people aged 70+ report more contacts with younger bands than those bands report with them. The largest error, about **3.99** million a day, is between `[70,inf)` and `[15,20)`.
> - The relative reciprocity error is about **0.25** — this raw survey matrix is far from reciprocal.
> - Mean contacts per person per day is about **11.07**.

## E. Rebin onto the model's age property

The model below has six age groups, named by their lower bounds:
`0, 5, 15, 25, 45, 65`. `polymod.rebin(age, population=UK_2006)` moves the
fifteen survey bands onto them: merging participant bands takes a
population-weighted mean of their rows, merging contact bands sums their
columns. `Rebin.between(...)` is the same aggregation as an object you can
inspect — two constant `(6, 15)` matrices, `rows` and `cols`, applied as
`rows @ values @ cols.T`. The left heatmap is `rows`; the right is the
rebinned matrix. The table compares mean contacts per person before and after.

In [ ]:
from summer4 import (
    Compartments,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
)
from summer4.epi import ContactAdaptation, ForceOfInfection, Rebin
from summer4.timevarying import step

age = Property("age", ("0", "5", "15", "25", "45", "65"))
plan = Rebin.between(polymod.bands, (0, 5, 15, 25, 45, 65, math.inf), population=UK_2006)
model_contacts = polymod.rebin(age, population=UK_2006)

weights = pd.DataFrame(
    plan.rows,
    index=pd.Index(model_contacts.labels, name="model band"),
    columns=pd.Index(polymod.labels, name="survey band"),
)
weights.plot(
    kind="imshow",
    title="E. Rebin.rows: how survey participant bands are averaged into model bands",
    labels={"x": "survey band", "y": "model band", "color": "weight"},
).show()
model_contacts.to_pandas().plot(
    kind="imshow",
    title="E. POLYMOD rebinned onto the model's six age groups",
    labels={"x": "contact band", "y": "participant band", "color": "contacts"},
).show()

pd.DataFrame(
    {
        "bands": [polymod.size, model_contacts.size],
        "mean contacts per person": [polymod.mean_contacts(), model_contacts.mean_contacts()],
    },
    index=["survey (15 bands)", "model (6 bands)"],
).round(4)

> **What to check**
>
> - In the `rows` heatmap each model band draws only on the survey bands inside it, and its weights sum to 1 across a row (for example `[5,15)` splits about 0.48 / 0.52 between `[5,10)` and `[10,15)`, their shares of the population). `[0,5)` is a single survey band (weight 1); `[65,inf)` averages `[65,70)` and `[70,inf)`, the latter weighted about 0.72 because it holds more people.
> - The rebinned heatmap keeps the survey's shape at coarser resolution: its brightest cell is `[5,15)` × `[5,15)` (about **7.9**), then `[15,25)` × `[15,25)` (about **5.9**).
> - Mean contacts per person are **the same** before and after (about **11.07**): rebinning moves contacts between cells, it does not create or lose them.

## F. Symmetrise against the population

`symmetrise()` replaces each pair of reported totals with their mean,
$c'_{ij} = (c_{ij} N_i + c_{ji} N_j) / (2 N_i)$, using the matrix's
`source_population` (here the UK population summed onto the model's bands by
the rebin). The bars compare the relative reciprocity error, and the mean
contacts, before and after.

In [ ]:
symmetric = model_contacts.symmetrise()
summary_f = pd.DataFrame(
    {
        "relative reciprocity error": [
            model_contacts.reciprocity_error().relative,
            symmetric.reciprocity_error().relative,
        ],
        "mean contacts per person": [model_contacts.mean_contacts(), symmetric.mean_contacts()],
    },
    index=pd.Index(["rebinned survey", "symmetrised"], name="matrix"),
)
fig = summary_f["relative reciprocity error"].plot(
    kind="bar", title="F. Symmetrising removes the reciprocity error"
)
fig.update_layout(yaxis_title="relative reciprocity error", showlegend=False)
fig.show()
summary_f.round(6)

> **What to check**
>
> - The rebinned survey's bar is clearly above zero (about **0.17**); the symmetrised bar is **0**.
> - Mean contacts per person are unchanged by symmetrising (both about **11.07**).

## G. Adapt to another population, under the convention you choose

Chapter 19 of the textbook carries the UK matrix to India's much younger
population. `adapt(population, kind=...)` offers the two conventions in the
literature. Both start from the ratio $r_j$ of each band's share of the target
population to its share of the survey's:

- `"density"` — $c'_{ij} = c_{ij} r_j$: the chance that two particular people
  meet is kept, so everyone reports more contacts with bands that are larger in
  India. This is the textbook's adjustment.
- `"frequency"` (the default) — the same, then each row rescaled back to the
  contacts per person it had: every band keeps its *number* of contacts; only
  who they are with shifts.

The line chart is contacts per person (`total()`) by band, on the fifteen survey
bands after symmetrising against the UK. The table is each result's
reciprocity error against the Indian population. India's population by age is
from the same textbook chapter.

In [ ]:
INDIA = np.array(
    [116879506, 117982127, 126155952, 126045565, 122504805, 117397268, 112176099,
     103460177, 90219895, 79440281, 68875962, 59256267, 48890529, 38260284, 52459668],
    dtype=float,
)
uk = polymod.symmetrise()
adapted = {
    "UK survey (symmetrised)": uk,
    "India, density": uk.adapt(INDIA, kind=ContactAdaptation.DENSITY),
    "India, frequency": uk.adapt(INDIA, kind=ContactAdaptation.FREQUENCY),
}
per_person = pd.DataFrame(
    {name: cm.total() for name, cm in adapted.items()},
    index=pd.Index(polymod.labels, name="band"),
)
fig = per_person.plot(title="G. Contacts per person after adapting to India", markers=True)
fig.update_layout(xaxis_title="participant band", yaxis_title="contacts per day", legend_title="")
fig.show()
pd.DataFrame(
    {
        "reciprocity error vs India": {
            name: round(cm.reciprocity_error(INDIA).relative, 4) for name, cm in adapted.items()
        },
        "mean contacts (India weights)": {
            name: round(cm.mean_contacts(INDIA), 2) for name, cm in adapted.items()
        },
    }
)

> **What to check**
>
> - The **frequency** line lies exactly on the UK line: every band keeps its contacts per person.
> - The **density** line is above the UK line for every band under 45 — most for school-age children, about **15.8 → 19.8** at `[5,10)` — and below it from 50 up (about **5.8 → 4.7** at `[70,inf)`). India has proportionally more young people and fewer old ones, so under density everyone meets more of the young.
> - In the table the density result is reciprocal against India (error **0**); the frequency result is not (about **0.05**), and the unadapted UK matrix is further off (about **0.2**). Mean contacts under India's weights rise to about **13.4** under density and stay at about **12.0** under frequency.

## H. Close schools from a date, and run an SEIR

The two-setting stack from part B is rebinned and symmetrised setting by
setting (`stack.rebin(...)`, `stack.symmetrise()`), then
`stack.scale(step(Time(), (50.0,), (1.0, 0.1)), settings=["school"])` keeps
school contacts at 100% until day 50 and at 10% from then on. The result is a
`ScaledContacts` — a rate expression — and `to_mixing()` hands it to the force
of infection, **with `normalize="none"`**: the row sums are the survey's contact
numbers and must survive, so `contact_rate` is the probability of transmission
per contact.

The model is an age-stratified SEIR with a 3-day latent and 4-day infectious
period, 1 in 100,000 people infectious on day 0, and a transmission probability
of 0.04 per contact. The top chart is the proportion of each age group
infectious over time, schools open (solid) and closed from day 50 (dashed); the
bar chart is the final proportion ever infected.

In [ ]:
model_stack = stack.rebin(age, population=UK_2006).symmetrise()
population = model_stack.total().source_population
state = Property("state", ("S", "E", "I", "R"))
pmap = PropertyMap.from_property(state).stratify(age)


def seir(mixing):
    model = FlowModel(pmap)
    infection = ForceOfInfection(
        "infection", infectious=state["I"], group_by=age, mixing=mixing, contact_rate=Param("per_contact")
    )
    model.add_flow(TransitionFlow("infection", state["S"], state["E"], infection))
    model.add_flow(TransitionFlow("onset", state["E"], state["I"], 1.0 / 3.0))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 4.0))
    return model.compile()


y0 = np.zeros(pmap.size)
for trait, n in zip(age.traits, population):
    y0[pmap.select(state["S"] & age[trait])] = n * (1.0 - 1e-5)
    y0[pmap.select(state["I"] & age[trait])] = n * 1e-5

days = np.arange(0.0, 201.0)
save = SavePlan(
    requests={
        "I": SaveRequest(Compartments(where=state["I"]), ts=days),
        "R": SaveRequest(Compartments(where=state["R"]), ts=days),
    }
)
closure = step(Time(), (50.0,), (1.0, 0.1))
scenarios = {
    "schools open": seir(model_stack.to_mixing()),
    "schools at 10% from day 50": seir(model_stack.scale(closure, settings=["school"]).to_mixing()),
}
runs = {
    name: model.run({"per_contact": 0.04}, y0, t0=0.0, t1=200.0, dt=0.25, save=save, solver="euler")
    for name, model in scenarios.items()
}

curves = []
for name, run in runs.items():
    infectious = pd.DataFrame(
        np.asarray(run["I"].values.data) / population, index=days, columns=list(model_contacts.labels)
    )
    curves.append(infectious.rename_axis("day").reset_index().melt("day", var_name="age", value_name="proportion infectious").assign(scenario=name))
curves = pd.concat(curves)
fig = curves.plot(
    x="day", y="proportion infectious", color="age", line_dash="scenario",
    title="H. Proportion infectious by age: schools open vs closed from day 50",
)
fig.add_vline(x=50.0, line_dash="dot", annotation_text="schools close")
fig.show()

attack = pd.DataFrame(
    {name: np.asarray(run["R"].values.data)[-1] / population for name, run in runs.items()},
    index=pd.Index(model_contacts.labels, name="age"),
)
fig = attack.plot(kind="bar", barmode="group", title="H. Proportion ever infected by day 200")
fig.update_layout(yaxis_title="proportion", legend_title="")
fig.show()
attack.round(3)

> **What to check**
>
> - Up to day 50 the solid and dashed curves are identical; the closure acts from the date, not before.
> - With schools open, `[5,15)` peaks first and highest. After day 50 the dashed `[5,15)` curve is well below the solid one, and every age group's dashed peak is later and lower.
> - In the bars the school-age group gains most from the closure (about **0.86 → 0.7** ever infected); the other groups fall too, because children infect adults.

## I. The school scale is calibratable

Replace the fixed step with a parameter: `stack.scale(Param("school"),
settings=["school"])`. The whole run is then a function of `school` that
`jax.jit` compiles once, `jax.vmap` sweeps, and `jax.grad` differentiates — what
a calibration needs to fit an intervention's size. The line is the overall
proportion ever infected as school contacts go from 0% to 100% of normal; the
table shows the gradient at 50% beside a finite difference.

In [ ]:
import jax
import jax.numpy as jnp

calibratable = seir(model_stack.scale(Param("school"), settings=["school"]).to_mixing())
final = SavePlan(requests={"R": SaveRequest(Compartments(where=state["R"]), ts=np.array([200.0]))})


@jax.jit
def ever_infected(school):
    run = calibratable.run(
        {"per_contact": 0.04, "school": school}, y0, t0=0.0, t1=200.0, dt=0.25, save=final, solver="euler"
    )
    return jnp.sum(run["R"].values.data[-1]) / population.sum()


levels = np.linspace(0.0, 1.0, 11)
sweep = pd.Series(np.asarray(jax.vmap(ever_infected)(jnp.asarray(levels))), index=pd.Index(levels, name="school contacts (share of normal)"))
fig = sweep.plot(title="I. Proportion ever infected against the school-contact scale", markers=True)
fig.update_layout(yaxis_title="proportion ever infected", showlegend=False)
fig.show()

h = 0.05
pd.Series(
    {
        "jax.grad at 0.5": float(jax.grad(ever_infected)(0.5)),
        "central difference at 0.5": float((ever_infected(0.5 + h) - ever_infected(0.5 - h)) / (2 * h)),
        "value at 1.0 (vs part H 'schools open')": float(ever_infected(1.0)),
        "part H 'schools open'": float(attack["schools open"] @ population / population.sum()),
    },
    name="value",
).to_frame()

> **What to check**
>
> - The line rises steadily from 0% to 100% school contacts.
> - The gradient at 0.5 is positive and agrees with the central difference to about two significant figures.
> - The value at 1.0 equals part H's schools-open run (the last two rows match): `Param("school") = 1` is the unscaled stack.

## J. The compiled model does not grow with the number of age bands

Every transformation above runs on the host, and the scaled matrix enters the
model as constant arrays plus one multiply and one add per scaled setting — no
loop over bands. The cell rebins POLYMOD onto 2, 3, 5, 8 and 15 bands, builds
the same school-closure SEIR for each, and counts the equations in the jaxpr of
its vector field (`jax.make_jaxpr`). Compartments grow with the bands; the
program XLA compiles should not.

In [ ]:
def vector_field_size(edges):
    bands_age = Property("age", tuple(str(int(e)) for e in edges[:-1]))
    rebinned = stack.rebin(bands_age, population=UK_2006).symmetrise()
    n = rebinned.total().source_population
    pm = PropertyMap.from_property(state).stratify(bands_age)
    model = FlowModel(pm)
    mixing = rebinned.scale(closure, settings=["school"]).to_mixing()
    infection = ForceOfInfection(
        "infection", infectious=state["I"], group_by=bands_age, mixing=mixing, contact_rate=Param("per_contact")
    )
    model.add_flow(TransitionFlow("infection", state["S"], state["E"], infection))
    model.add_flow(TransitionFlow("onset", state["E"], state["I"], 1.0 / 3.0))
    model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 4.0))
    compiled = model.compile()
    y = np.zeros(pm.size)
    for trait, size in zip(bands_age.traits, n):
        y[pm.select(state["S"] & bands_age[trait])] = size
    params = {"per_contact": 0.04}
    jaxpr = jax.make_jaxpr(lambda t: jnp.asarray(compiled.vector_field(t, jnp.asarray(y), params)))(60.0)
    return {"compartments": pm.size, "vector-field equations": len(jaxpr.jaxpr.eqns)}


resolutions = {
    2: (0, 20, math.inf),
    3: (0, 20, 60, math.inf),
    5: (0, 5, 20, 40, 60, math.inf),
    8: (0, 5, 10, 20, 30, 40, 50, 60, math.inf),
    15: BANDS,
}
sizes = pd.DataFrame({k: vector_field_size(edges) for k, edges in resolutions.items()}).T.rename_axis("age bands")
fig = sizes.plot(title="J. Compartments grow with age bands; the vector-field program does not", markers=True)
fig.update_layout(yaxis_title="count", legend_title="")
fig.show()
sizes

> **What to check**
>
> - The compartments line rises (4 per band: 8 → 60).
> - The vector-field equations line is **flat**: the same count at 2 bands as at 15.